# XFakeNet – Week 2 on Google Colab

Continues from Week 1 (it reuses the data and baseline model saved in your Google Drive).

| Section | What it does | Time on T4 |
|---|---|---|
| A | **Module 2:** CLIP C-score + threshold calibration on the FakeNewsNet pairs | ~5 min |
| B | **Gemini paraphrases** (15% of corpus + 500 test statements) | ~15–30 min |
| C | **Model v1:** three-class RoBERTa, class-weighted loss, learning-rate sweep | ~25–40 min |
| D | **TextFooler** adversarial examples (15% of corpus + 500 test statements) | ~30–90 min |
| E | **Results for your mentor** + download zip | seconds |

**Before you start**
1. `Runtime → Change runtime type → T4 GPU → Save`
2. **Gemini key (for section B):** get a free key at https://aistudio.google.com/apikey. In Colab click the **key icon** on the left sidebar → **Add new secret** → Name `GEMINI_API_KEY`, Value = your key → turn on **Notebook access**. (If you skip this, the notebook will ask you to paste the key.)
3. `Runtime → Run all`, click **Allow** for Google Drive, and **keep this tab open**.

Every long step saves its progress to Drive. If Colab disconnects, just `Run all` again: finished work is skipped.

## 1. GPU, Google Drive, and Week 1 files

In [ ]:
import os, subprocess
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE – Runtime > Change runtime type > T4 GPU, then Run all again")

from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/xfakenet"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

need = ["data/processed/corpus_train.csv", "data/processed/fnn_pairs.csv", "runs/baseline/best/config.json"]
missing = [f for f in need if not os.path.exists(f)]
if missing:
    raise SystemExit(f"Week 1 files missing: {missing}. Run the Week 1 notebook first.")
print("Week 1 data and baseline model found in", WORK)

## 2. Write the project code
(These cells just save the scripts into the project folder.)

In [ ]:
import os
for d in ['configs','src/xfakenet','scripts','docs','logs','data/augment','results']:
    os.makedirs(d, exist_ok=True)
print('folders ready')

In [ ]:
%%writefile configs/week1.yaml
seed: 42
paths:
  raw: data/raw
  processed: data/processed
  images: data/images
  runs: runs

liar:
  dir: data/raw/liar            # expects train.tsv / valid.tsv / test.tsv

extra_classes:
  enabled: false      # three-class setup: LIAR only
  source: tshp17
  file: data/raw/tshp17/fulltrain.csv   # columns: label,text (no header)
  test_file: data/raw/tshp17/balancedtest.csv
  use_hoax_trusted: false
  # Articles are long; LIAR statements are ~18 words. Truncate to the first
  # sentences up to this many words so length is not a shortcut for the class.
  max_words: 40
  # Cap per class so Satire/Propaganda stay minority classes (as in the outline).
  max_per_class: 1500

fakenewsnet:
  dir: data/raw/FakeNewsNet/dataset
  sample_size: 1000            # articles to try for images (balanced fake/real)
  min_image_side: 100          # px; smaller images (icons, pixels) are discarded
  workers: 16
  timeout: 10
  use_wayback: true

baseline:
  model: roberta-base
  max_len: 64
  epochs: 4
  batch_size: 32
  lr: 2.0e-5
  warmup_ratio: 0.06
  weight_decay: 0.01
  fp16: true

# ---------------- Week 2 ----------------
augmentation:                  # final training mix (week 3): 70% original / 15% / 15%
  original_share: 70
  textfooler_share: 15
  paraphrase_share: 15

cscore:
  clip_model: openai/clip-vit-base-patch32
  batch_size: 64
  percentile: 95               # alternative threshold: 95th percentile of aligned C-scores (valid)
  out_dir: results/cscore

paraphrase:
  models: [gemini-3.5-flash-lite, gemini-3.8-flash, gemini-3.1-flash-lite, gemini-2.5-flash]
  batch_size: 20               # statements per API call
  temperature: 0.9
  sleep_s: 4                   # pause between calls (free-tier rate limits)
  test_n: 500
  out_train: data/augment/paraphrase_train.jsonl
  out_test: data/augment/paraphrase_test.jsonl

textfooler:
  victim_dir: runs/baseline/best
  embeddings_url: https://github.com/nmrksic/counter-fitting/raw/master/word_vectors/counter-fitted-vectors.txt.zip
  embeddings_path: data/raw/counter-fitted-vectors.txt.zip
  sim_model: sentence-transformers/all-MiniLM-L6-v2
  k: 50                        # synonyms per word (paper: 50)
  sim_threshold: 0.80
  max_change: 0.25             # at most 25% of words replaced
  query_budget: 2000
  test_n: 500
  out_train: data/augment/textfooler_train.jsonl
  out_test: data/augment/textfooler_test.jsonl

model_v1:
  lrs: [1.0e-5, 2.0e-5, 3.0e-5]
  epochs: 5

# ---------------- Week 3 ----------------
xfakenet:                      # RoBERTa + jointly trained token/segment attention
  encoder: roberta-base
  max_len: 64
  max_segments: 8
  att_dim: 256
  epochs: 4
  batch_size: 32
  lr: 2.0e-5                   # encoder
  head_lr: 1.0e-4              # attention layers + classifier
  warmup_ratio: 0.06
  weight_decay: 0.01
  highlight_threshold: 0.65    # outline: tokens above T = 0.65 are highlighted

modality:
  C: 0.1                       # regularisation of the text/image logistic models (389 items)

week3:
  out_dir: results/week3

# ---------------- Week 4 ----------------
week4:
  out_dir: results/week4

deception:
  n_mismatched: 200            # shuffled pairs added to the 389 real ones for labelling
  batch_size: 5                # image-text pairs per Gemini call
  image_side: 512
  sleep_s: 4
  C: 0.5
  min_per_class: 5
  handcheck_n: 100
  labels: data/deception/gemini_labels.jsonl
  features: data/deception/clip_features.npz
  oof: data/deception/classifier_oof.csv
  model: runs/deception_clf.joblib
  handcheck_sample: data/deception/handcheck_sample.csv
  handcheck_results: data/deception/handcheck_results.csv

pipeline:
  modality_model: runs/modality.joblib

explain:                       # faithful explanations (week 4 fix)
  permutations_gpu: 10
  permutations_cpu: 6
  cpu_statements: 400
  chunk: 50

In [ ]:
%%writefile configs/label_mapping.yaml
# XFakeNet five-class label mapping (frozen 2026-10-21, Week 1)
# Every later step (training, augmentation, evaluation) reads this file.
# Change it only with a note in docs/label_mapping.md and a full retrain.

# Three-class setup (changed 20 Oct 2026): Satire/Propaganda dropped because the
# only available source (TSHP-17) taught the model website style, not the class
# (Propaganda F1 0.99 on same-site validation, 0.08 on unseen-site test).
# For the old five-class setup use:
#   classes: [Fabricated, Misleading, Satire, Propaganda, Credible]
#   and set extra_classes.enabled: true in week1.yaml
classes: [Fabricated, Misleading, Credible]   # index order = model label ids

liar:
  # LIAR's six PolitiFact truth ratings -> XFakeNet classes
  # Keys are quoted on purpose: unquoted true/false become YAML booleans.
  "pants-fire":  Fabricated
  "false":       Fabricated
  "barely-true": Misleading
  "half-true":   Misleading
  "mostly-true": Credible
  "true":        Credible

# LIAR has no Satire or Propaganda signal. These two classes come from an
# external labelled source (default: TSHP-17, Rashkin et al. 2017).
tshp17:
  # TSHP-17 integer labels: 1=Satire 2=Hoax 3=Propaganda 4=Trusted
  1: Satire
  3: Propaganda
  # 2 (Hoax) and 4 (Trusted) are NOT used by default: they would add
  # long-article Fabricated/Credible examples whose style differs from LIAR.
  # Set use_hoax_trusted: true in configs/week1.yaml to include them.
  2: Fabricated
  4: Credible

fakenewsnet:
  # Binary only; used for C-score calibration/evaluation, not five-class training.
  "fake": Fabricated
  "real": Credible

In [ ]:
%%writefile src/xfakenet/__init__.py
# package marker

In [ ]:
%%writefile src/xfakenet/common.py
"""Shared helpers: config loading, label mapping, seeding, text cleanup."""
from __future__ import annotations

import json
import random
import re
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[2]


def load_yaml(path: str | Path) -> dict:
    with open(ROOT / path if not Path(path).is_absolute() else path) as f:
        return yaml.safe_load(f)


def load_config() -> dict:
    return load_yaml("configs/week1.yaml")


def load_mapping() -> dict:
    return load_yaml("configs/label_mapping.yaml")


def class_names() -> list[str]:
    return load_mapping()["classes"]


def label2id() -> dict[str, int]:
    return {c: i for i, c in enumerate(class_names())}


def p(rel: str | Path) -> Path:
    """Resolve a repo-relative path."""
    path = ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def set_seed(seed: int) -> None:
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


_WS = re.compile(r"\s+")
_SENT = re.compile(r"(?<=[.!?])\s+")


def clean_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.replace("“", '"').replace("”", '"').replace("’", "'")
    return _WS.sub(" ", s).strip()


def truncate_words(text: str, max_words: int) -> str:
    """Keep whole leading sentences up to max_words; hard-cut if the first sentence is longer."""
    text = clean_text(text)
    out, n = [], 0
    for sent in _SENT.split(text):
        w = sent.split()
        if n + len(w) > max_words:
            if not out:
                out.append(" ".join(w[:max_words]))
            break
        out.append(sent)
        n += len(w)
    return " ".join(out)


def write_json(obj, path: str | Path) -> None:
    with open(p(path), "w") as f:
        json.dump(obj, f, indent=2, default=str)

In [ ]:
%%writefile src/xfakenet/textfooler.py
"""TextFooler (Jin et al., AAAI 2020, "Is BERT Really Robust?") without the TextAttack dependency.

Algorithm, as in the paper:
1. Rank words by importance: how much the true-class probability drops when the word is deleted.
2. For each word in that order, take its nearest neighbours in counter-fitted word
   embeddings (synonyms, cos >= 0.5).
3. Keep candidates that preserve the part of speech and keep the sentence meaning
   (sentence-embedding cosine >= threshold to the ORIGINAL text).
4. If a candidate flips the prediction, return the one closest in meaning (success).
   Otherwise keep the candidate that lowers the true-class probability most and continue.

Differences from the paper (documented for the report):
- Sentence similarity uses all-MiniLM-L6-v2 instead of the Universal Sentence Encoder.
- Model queries are batched and constraints are checked lazily (only on candidates
  the model output makes useful), which gives the same result order much faster.
- At most `max_change` (default 25%) of the words may be replaced.
- v2 quality filters (added after inspecting v1 output, where 28% of examples swapped a
  person/party/programme name and so changed the factual claim):
    * proper nouns, numbers and acronyms are never replaced (POS tags NNP/NNPS/CD, any
      capitalised word that is not sentence-initial, all-caps tokens);
    * candidates that are WordNet antonyms of the original word are rejected;
    * candidates on an offensive-word blocklist are rejected.

The class only needs four callables, so it can be unit-tested without a GPU.
"""
from __future__ import annotations

import math
import re
from typing import Callable, Sequence

import numpy as np

# Stopwords are never replaced (subset of the list used by TextFooler).
STOP = set("""a about above across after afterwards again against ain all almost alone along already also
although am among amongst an and another any anyhow anyone anything anyway anywhere are aren aren't around as
at back be became because become becomes becoming been before beforehand behind being below beside besides
between beyond both but by can cannot could couldn couldn't d didn didn't doesn doesn't don don't down due during
either else elsewhere empty enough even ever everyone everything everywhere except first for former formerly from
hadn hadn't hasn hasn't haven haven't he hence her here hereafter hereby herein hereupon hers herself him himself
his how however hundred i if in indeed into is isn isn't it it's its itself just latter latterly least ll may me
meanwhile mightn mightn't mine more moreover most mostly must mustn mustn't my myself namely needn needn't neither
never nevertheless next no nobody none noone nor not nothing now nowhere o of off on once one only onto or other
others otherwise our ours ourselves out over per please s same shan shan't she she's should've shouldn shouldn't
somehow something sometime somewhere such t than that that'll the their theirs them themselves then thence there
thereafter thereby therefore therein thereupon these they this those through throughout thru thus to too toward
towards under unless until up upon used ve was wasn wasn't we were weren weren't what whatever when whence
whenever where whereafter whereas whereby wherein whereupon wherever whether which while whither who whoever whole
whom whose why with within without won won't would wouldn wouldn't y yet you you'd you'll you're you've your yours
yourself yourselves says said""".split())

_AFFIX = re.compile(r"^(\W*)(.*?)(\W*)$")

# Candidates never used as replacements (offensive words that appear in embedding neighbourhoods).
BLOCKLIST = set("""slut sluts whore whores bitch bitches bastard bastards fuck fucking fucked shit shitty
crap cunt dick dicks cock pussy asshole arse arsehole retard retarded faggot fag nigger nigga
negro spic chink kike wetback tranny dyke hooker prostitute rapist""".split())


def split_tok(tok: str) -> tuple[str, str, str]:
    m = _AFFIX.match(tok)
    return m.group(1), m.group(2), m.group(3)


def match_case(src: str, cand: str) -> str:
    if src.isupper() and len(src) > 1:
        return cand.upper()
    if src[:1].isupper():
        return cand[:1].upper() + cand[1:]
    return cand


class TextFooler:
    def __init__(self,
                 predict: Callable[[Sequence[str]], np.ndarray],          # texts -> probs [n, C]
                 neighbours: Callable[[str], list[str]],                   # word -> synonyms
                 similarity: Callable[[str, Sequence[str]], np.ndarray],   # orig, texts -> cos [n]
                 pos_ok: Callable[[list[str], int, str], bool],            # words, idx, cand -> ok
                 protect: Callable[[list[str]], set] | None = None,        # words -> indices never replaced
                 banned: Callable[[str, str], bool] | None = None,         # (orig word, cand) -> reject?
                 sim_threshold: float = 0.80,
                 max_change: float = 0.25,
                 query_budget: int = 2000,
                 max_checks: int = 8):
        self.predict, self.neighbours = predict, neighbours
        self.similarity, self.pos_ok = similarity, pos_ok
        self.sim_threshold, self.max_change = sim_threshold, max_change
        self.query_budget, self.max_checks = query_budget, max_checks
        self.protect, self.banned = protect, banned

    def _ok(self, orig: str, words: list[str], i: int, cand_core: str, text: str) -> tuple[bool, float]:
        if not self.pos_ok(words, i, cand_core):
            return False, 0.0
        s = float(self.similarity(orig, [text])[0])
        return s >= self.sim_threshold, s

    def attack(self, text: str, label: int) -> dict:
        toks = text.split()
        parts = [split_tok(t) for t in toks]
        cores = [c for _, c, _ in parts]
        p0 = self.predict([text])[0]
        q = 1
        if int(np.argmax(p0)) != label:
            return {"status": "skipped", "queries": q}

        protected = set(self.protect(cores)) if self.protect else set()
        elig = [i for i, c in enumerate(cores)
                if c.isalpha() and len(c) >= 3 and c.lower() not in STOP
                and i not in protected
                and not (i > 0 and c[:1].isupper())        # capitalised mid-sentence: a name
                and not c.isupper()]                       # acronyms (CDC, NATO)
        if not elig:
            return {"status": "failed", "queries": q, "reason": "no_eligible_words"}

        # 1. word importance by deletion
        dels = [" ".join(toks[:i] + toks[i + 1:]) for i in elig]
        pd_ = self.predict(dels)
        q += len(dels)
        imp = p0[label] - pd_[:, label]
        flip = pd_.argmax(1) != label
        imp = imp + flip * (pd_.max(1) - p0[pd_.argmax(1)])   # paper's extra term when deletion flips
        order = [elig[k] for k in np.argsort(-imp)]

        cur = list(toks)
        cur_p = float(p0[label])
        changed, max_changes = 0, max(1, math.ceil(self.max_change * len(toks)))

        for i in order:
            if changed >= max_changes or q >= self.query_budget:
                break
            pre, core, post = parts[i]
            syns = [s for s in self.neighbours(core.lower())
                    if s.lower() != core.lower() and s.lower() not in BLOCKLIST
                    and not (self.banned and self.banned(core.lower(), s.lower()))]
            if not syns:
                continue
            cands = [match_case(core, s) for s in syns]
            texts = []
            for c in cands:
                t = list(cur)
                t[i] = pre + c + post
                texts.append(" ".join(t))
            pr = self.predict(texts)
            q += len(texts)
            words_now = [split_tok(t)[1] for t in cur]

            flipped = np.where(pr.argmax(1) != label)[0]
            if len(flipped):
                sims = self.similarity(text, [texts[k] for k in flipped])
                for k_idx in np.argsort(-sims):
                    k = flipped[k_idx]
                    if sims[k_idx] >= self.sim_threshold and self.pos_ok(words_now, i, cands[k]):
                        out = texts[k]
                        return {"status": "success", "text": out, "queries": q,
                                "words_changed": changed + 1,
                                "similarity": round(float(sims[k_idx]), 4),
                                "new_label_id": int(pr[k].argmax()),
                                "orig_conf": round(float(p0[label]), 4),
                                "adv_conf": round(float(pr[k][label]), 4)}

            # no valid flip: keep the best probability-lowering candidate that passes the constraints
            for k in np.argsort(pr[:, label])[: self.max_checks]:
                if pr[k, label] >= cur_p:
                    break
                ok, _ = self._ok(text, words_now, i, cands[k], texts[k])
                if ok:
                    cur[i] = pre + cands[k] + post
                    cur_p = float(pr[k, label])
                    changed += 1
                    break
        return {"status": "failed", "queries": q, "words_changed": changed}


# ---------- real components (used on Colab / the GPU server) ----------

class CounterFitted:
    """Nearest neighbours in counter-fitted vectors (Mrksic et al., 2016), as used by TextFooler."""

    def __init__(self, path: str, k: int = 50, min_cos: float = 0.5):
        words, vecs = [], []
        opener = open
        if path.endswith(".zip"):
            import io
            import zipfile
            z = zipfile.ZipFile(path)
            name = [n for n in z.namelist() if n.endswith(".txt")][0]
            fh = io.TextIOWrapper(z.open(name), encoding="utf-8")
        else:
            fh = opener(path, encoding="utf-8")
        for line in fh:
            w, *v = line.rstrip().split(" ")
            if len(v) < 10:
                continue
            words.append(w)
            vecs.append(np.asarray(v, dtype=np.float32))
        m = np.vstack(vecs)
        self.M = m / np.linalg.norm(m, axis=1, keepdims=True).clip(1e-8)
        self.words, self.idx = words, {w: i for i, w in enumerate(words)}
        self.k, self.min_cos, self.cache = k, min_cos, {}

    def __call__(self, w: str) -> list[str]:
        if w in self.cache:
            return self.cache[w]
        i = self.idx.get(w)
        if i is None:
            self.cache[w] = []
            return []
        sims = self.M @ self.M[i]
        kk = min(self.k + 1, len(sims) - 1)
        top = np.argpartition(-sims, kk)[: kk + 1]
        top = top[np.argsort(-sims[top])]
        res = [self.words[j] for j in top if j != i and sims[j] >= self.min_cos][: self.k]
        self.cache[w] = res
        return res


class WordNetSyn:
    """Fallback synonym source if the counter-fitted vectors cannot be downloaded."""

    def __init__(self, k: int = 50):
        from nltk.corpus import wordnet as wn
        self.wn, self.k, self.cache = wn, k, {}

    def __call__(self, w: str) -> list[str]:
        if w not in self.cache:
            out = []
            for s in self.wn.synsets(w):
                for l in s.lemma_names():
                    l = l.replace("_", " ")
                    if " " not in l and l.lower() != w and l not in out:
                        out.append(l)
            self.cache[w] = out[: self.k]
        return self.cache[w]


def make_pos_ok(window: int = 3):
    import nltk

    def coarse(tag: str) -> str:
        return tag[0] if tag[0] in "NVJR" else tag

    def pos_ok(words: list[str], i: int, cand: str) -> bool:
        lo, hi = max(0, i - window), min(len(words), i + window + 1)
        a = [w if w else "," for w in words[lo:hi]]
        b = list(a)
        b[i - lo] = cand
        ta, tb = nltk.pos_tag(a), nltk.pos_tag(b)
        return coarse(ta[i - lo][1]) == coarse(tb[i - lo][1])

    return pos_ok


def make_protect():
    """Indices of proper nouns and numbers (never replaced)."""
    import nltk

    def protect(words: list[str]) -> set:
        tags = nltk.pos_tag([w if w else "," for w in words])
        return {i for i, (_, t) in enumerate(tags) if t in ("NNP", "NNPS", "CD")}

    return protect


def make_banned():
    """Reject WordNet antonyms of the original word (e.g. died -> lives)."""
    from nltk.corpus import wordnet as wn
    cache: dict[str, set] = {}

    def antonyms(w: str) -> set:
        if w not in cache:
            out = set()
            for syn in wn.synsets(w):
                for lem in syn.lemmas():
                    for a in lem.antonyms():
                        out.add(a.name().lower())
                        for rel in a.derivationally_related_forms():
                            out.add(rel.name().lower())
            cache[w] = out
        return cache[w]

    def banned(orig: str, cand: str) -> bool:
        if cand in antonyms(orig) or orig in antonyms(cand):
            return True
        # also compare base forms (died/die vs lives/live)
        base_o, base_c = wn.morphy(orig) or orig, wn.morphy(cand) or cand
        return base_c in antonyms(base_o) or base_o in antonyms(base_c)

    return banned


def make_similarity(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer(model_name)
    cache: dict[str, np.ndarray] = {}

    def sim(orig: str, texts: Sequence[str]) -> np.ndarray:
        if orig not in cache:
            if len(cache) > 50:
                cache.clear()
            cache[orig] = st.encode([orig], normalize_embeddings=True)[0]
        e = st.encode(list(texts), normalize_embeddings=True, batch_size=64)
        return e @ cache[orig]

    return sim


def make_predict(model_dir: str, max_len: int = 64, batch_size: int = 64):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(dev).eval()

    @torch.no_grad()
    def predict(texts: Sequence[str]) -> np.ndarray:
        out = []
        for s in range(0, len(texts), batch_size):
            enc = tok(list(texts[s:s + batch_size]), truncation=True, max_length=max_len,
                      padding=True, return_tensors="pt").to(dev)
            with torch.autocast(device_type="cuda", enabled=dev == "cuda"):
                logits = model(**enc).logits
            out.append(torch.softmax(logits.float(), -1).cpu().numpy())
        return np.concatenate(out)

    return predict

In [ ]:
%%writefile scripts/02_prepare_liar.py
"""Load LIAR, clean it, and apply the five-class mapping.

Reads the original TSVs from data/raw/liar/{train,valid,test}.tsv.
(The Hugging Face copy `ucsbnlp/liar` is script-based and no longer loads.)

Output: data/processed/liar_{train,valid,test}.csv with columns
    id, text, liar_label, label, source, split, speaker, party, subject, context
"""
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import clean_text, load_config, load_mapping, p, write_json  # noqa: E402

COLS = ["id", "liar_label", "text", "subject", "speaker", "job", "state", "party",
        "barely_true_ct", "false_ct", "half_true_ct", "mostly_true_ct", "pants_fire_ct", "context"]
SPLITS = {"train": "train.tsv", "valid": "valid.tsv", "test": "test.tsv"}


def load_tsv(d: Path) -> dict[str, pd.DataFrame]:
    out = {}
    for split, fn in SPLITS.items():
        df = pd.read_csv(d / fn, sep="\t", header=None, names=COLS, quoting=3,
                         dtype=str, keep_default_na=False)
        out[split] = df
    return out


def main() -> None:
    cfg, mapping = load_config(), load_mapping()["liar"]
    d = p(cfg["liar"]["dir"])
    if not (d / "train.tsv").exists():
        sys.exit(f"[liar] {d}/train.tsv not found. Download liar_dataset.zip "
                 "(UCSB, or search 'LIAR dataset' on Kaggle) and unzip it into that folder.")
    raw = load_tsv(d)

    stats = {}
    for split, df in raw.items():
        df = df.copy()
        df["liar_label"] = df["liar_label"].str.strip().str.lower()
        unknown = set(df["liar_label"]) - set(mapping)
        if unknown:
            raise ValueError(f"Unmapped LIAR labels in {split}: {unknown}")
        df["text"] = df["text"].map(clean_text)
        n0 = len(df)
        df = df[df["text"].str.len() > 0].drop_duplicates(subset=["text"])
        df["label"] = df["liar_label"].map(mapping)
        df["source"] = "liar"
        df["split"] = split
        keep = ["id", "text", "liar_label", "label", "source", "split",
                "speaker", "party", "subject", "context"]
        df = df[[c for c in keep if c in df.columns]]
        df.to_csv(p(f"{cfg['paths']['processed']}/liar_{split}.csv"), index=False)
        stats[split] = {
            "rows_raw": n0,
            "rows_clean": len(df),
            "liar_labels": df["liar_label"].value_counts().to_dict(),
            "xfakenet_labels": df["label"].value_counts().to_dict(),
            "mean_words": round(df["text"].str.split().str.len().mean(), 1),
        }
        print(f"[liar] {split}: {n0} -> {len(df)} rows | {stats[split]['xfakenet_labels']}")
    write_json(stats, f"{cfg['paths']['processed']}/liar_stats.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/03_prepare_extra_classes.py
"""Build Satire and Propaganda examples from TSHP-17 (Rashkin et al., 2017).

LIAR's six truth ratings contain no satire or propaganda signal, so these two
classes need an external source. TSHP-17 labels news articles as
1=Satire, 2=Hoax, 3=Propaganda, 4=Trusted.

To keep the style close to LIAR's short statements, each article is cut to its
leading sentences (<= max_words) so the model cannot cheat on length.

Input : data/raw/tshp17/fulltrain.csv (+ balancedtest.csv), no header: label,text
Output: data/processed/extra_{train,valid,test}.csv
"""
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, load_mapping, p, truncate_words, write_json  # noqa: E402


def cap_per_class(d: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    parts = [g.sample(min(len(g), n), random_state=seed) for _, g in d.groupby("label")]
    return pd.concat(parts) if parts else d


def read(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=["tshp_label", "text"], usecols=[0, 1], dtype=str)
    # tolerate a header row or stray lines: keep rows whose label is 1-4
    df["tshp_label"] = pd.to_numeric(df["tshp_label"], errors="coerce")
    df = df[df["tshp_label"].isin([1, 2, 3, 4])].copy()
    df["tshp_label"] = df["tshp_label"].astype(int)
    return df


def main() -> None:
    cfg = load_config()
    ec = cfg["extra_classes"]
    out_dir = cfg["paths"]["processed"]
    if not ec["enabled"]:
        print("[extra] disabled in config; skipping")
        return
    f = p(ec["file"])
    if not f.exists():
        print(f"[extra] {f} not found. Satire/Propaganda will be EMPTY until you add a source.\n"
              "        See docs/label_mapping.md, section 'Getting Satire and Propaganda data'.")
        return

    mapping = {int(k): v for k, v in load_mapping()["tshp17"].items()}
    keep = {1, 3} | ({2, 4} if ec["use_hoax_trusted"] else set())

    df = read(f)
    test_f = p(ec["test_file"])
    df_test = read(test_f) if test_f.exists() else None

    def prep(d: pd.DataFrame) -> pd.DataFrame:
        d = d[d["tshp_label"].isin(keep)].copy()
        d["text"] = d["text"].map(lambda t: truncate_words(t, ec["max_words"]))
        d = d[d["text"].str.split().str.len() >= 5].drop_duplicates(subset=["text"])
        d["label"] = d["tshp_label"].map(mapping)
        d["source"] = "tshp17"
        return d

    df = prep(df)
    # cap per class, reproducibly
    df = cap_per_class(df, ec["max_per_class"], cfg["seed"])

    if df_test is not None:
        test = cap_per_class(prep(df_test), max(1, ec["max_per_class"] // 8), cfg["seed"])
        train, valid = train_test_split(df, test_size=0.1, stratify=df["label"], random_state=cfg["seed"])
    else:
        train, rest = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=cfg["seed"])
        valid, test = train_test_split(rest, test_size=0.5, stratify=rest["label"], random_state=cfg["seed"])

    stats = {}
    for split, d in [("train", train), ("valid", valid), ("test", test)]:
        d = d.copy()
        d["split"] = split
        d["id"] = [f"tshp-{split}-{i}" for i in range(len(d))]
        d[["id", "text", "label", "source", "split"]].to_csv(p(f"{out_dir}/extra_{split}.csv"), index=False)
        stats[split] = d["label"].value_counts().to_dict()
        print(f"[extra] {split}: {stats[split]}")
    write_json(stats, f"{out_dir}/extra_stats.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/04_build_corpus.py
"""Merge LIAR and the extra-class source into one five-class corpus.

Output: data/processed/corpus_{train,valid,test}.csv  (columns: id,text,label,label_id,source)
        data/processed/corpus_stats.json  (class counts + class weights for the loss)
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import class_names, label2id, load_config, p, write_json  # noqa: E402


def main() -> None:
    cfg = load_config()
    d = cfg["paths"]["processed"]
    l2i, classes = label2id(), class_names()
    stats = {}
    for split in ["train", "valid", "test"]:
        parts = [pd.read_csv(p(f"{d}/liar_{split}.csv"))]
        extra = p(f"{d}/extra_{split}.csv")
        if cfg["extra_classes"]["enabled"] and extra.exists():
            parts.append(pd.read_csv(extra))
        df = pd.concat(parts, ignore_index=True)[["id", "text", "label", "source"]]
        df = df[df["label"].isin(classes)]   # drop anything outside the active class list
        df = df.drop_duplicates(subset=["text"])
        df["label_id"] = df["label"].map(l2i)
        df = df.sample(frac=1.0, random_state=cfg["seed"]).reset_index(drop=True)
        df.to_csv(p(f"{d}/corpus_{split}.csv"), index=False)
        counts = {c: int((df["label"] == c).sum()) for c in classes}
        stats[split] = {"n": len(df), "counts": counts,
                        "by_source": df.groupby(["source", "label"]).size().unstack(fill_value=0).to_dict()}
        print(f"[corpus] {split}: n={len(df)} {counts}")

    # Inverse-frequency class weights from TRAIN, normalised to mean 1. Absent classes get 0.
    c = np.array([stats["train"]["counts"][k] for k in classes], dtype=float)
    present = c > 0
    w = np.zeros_like(c)
    w[present] = c[present].sum() / (present.sum() * c[present])
    w[present] /= w[present].mean()
    stats["class_weights"] = dict(zip(classes, np.round(w, 4).tolist()))
    missing = [k for k, n in zip(classes, c) if n == 0]
    if missing:
        stats["warning"] = f"No training data for {missing}; model will never predict them."
        print(f"[corpus] WARNING: no training data for {missing}")
    print(f"[corpus] class weights: {stats['class_weights']}")
    write_json(stats, f"{d}/corpus_stats.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/05_fetch_fnn_images.py
"""Download FakeNewsNet article images and count usable headline-image pairs.

For each sampled article we fetch the page (live, then the Wayback Machine if the
live link is dead), take og:image / twitter:image (or the first large <img>),
download it and check it opens and is at least min_image_side px on each side.

Resumable: already-processed ids in the manifest are skipped, so you can
kill and restart it freely.

Input : data/raw/FakeNewsNet/dataset/{politifact,gossipcop}_{fake,real}.csv
Output: data/images/<id>.jpg
        data/processed/fnn_manifest.jsonl     (one row per article, with status)
        data/processed/fnn_pairs.csv          (usable pairs only: id,title,label,image_path,source_url)
        data/processed/fnn_image_summary.json (the counts for the week-1 report)

Usage : python scripts/05_fetch_fnn_images.py [--limit N] [--summary-only]
"""
from __future__ import annotations

import argparse
import io
import json
import sys
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import clean_text, load_config, p, write_json  # noqa: E402

UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) XFakeNet-research/0.1"}
_lock = threading.Lock()


def load_fnn(d: Path) -> pd.DataFrame:
    frames = []
    for site in ["politifact", "gossipcop"]:
        for lab in ["fake", "real"]:
            f = d / f"{site}_{lab}.csv"
            if f.exists():
                df = pd.read_csv(f, dtype=str, keep_default_na=False)
                df["site"], df["fnn_label"] = site, lab
                frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No FakeNewsNet CSVs in {d}. Run scripts/01_download_data.sh first.")
    df = pd.concat(frames, ignore_index=True)
    df["title"] = df["title"].map(clean_text)
    df = df[(df["news_url"].str.len() > 0) & (df["title"].str.len() > 0)]
    return df.drop_duplicates(subset=["id"])


def sample_balanced(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    """Balanced over fake/real and over sites, as far as the data allows."""
    groups = df.groupby(["site", "fnn_label"])
    per = max(1, n // groups.ngroups)
    parts = [g.sample(min(len(g), per), random_state=seed) for _, g in groups]
    out = pd.concat(parts)
    if len(out) < n:  # top up from whatever is left
        rest = df.drop(out.index)
        out = pd.concat([out, rest.sample(min(len(rest), n - len(out)), random_state=seed)])
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


def norm_url(u: str) -> str:
    u = u.strip()
    return u if u.startswith(("http://", "https://")) else "http://" + u


def get(url: str, timeout: int) -> requests.Response | None:
    try:
        r = requests.get(url, headers=UA, timeout=timeout, allow_redirects=True)
        return r if r.status_code == 200 else None
    except requests.RequestException:
        return None


def wayback(url: str, timeout: int) -> str | None:
    r = get(f"https://archive.org/wayback/available?url={url}&timestamp=20180101", timeout)
    if r is None:
        return None
    try:
        snap = r.json()["archived_snapshots"]["closest"]
        return snap["url"] if snap.get("available") else None
    except (KeyError, ValueError):
        return None


def find_image_url(html: str, base: str) -> str | None:
    soup = BeautifulSoup(html, "html.parser")
    for attr, key in [("property", "og:image"), ("name", "twitter:image"),
                      ("property", "og:image:url"), ("name", "twitter:image:src")]:
        tag = soup.find("meta", attrs={attr: key})
        if tag and tag.get("content"):
            return urljoin(base, tag["content"].strip())
    for img in soup.find_all("img", src=True):
        src = img["src"]
        if any(s in src.lower() for s in ("logo", "icon", "sprite", "pixel", ".svg", "avatar")):
            continue
        return urljoin(base, src)
    return None


def process(row: dict, cfg: dict) -> dict:
    from PIL import Image  # local import keeps worker startup cheap

    fc = cfg["fakenewsnet"]
    url = norm_url(row["news_url"])
    rec = {"id": row["id"], "title": row["title"], "fnn_label": row["fnn_label"],
           "site": row["site"], "news_url": url, "status": None, "via": None}

    page, via = get(url, fc["timeout"]), "live"
    if page is None and fc["use_wayback"]:
        snap = wayback(url, fc["timeout"])
        page, via = (get(snap, fc["timeout"]), "wayback") if snap else (None, None)
    if page is None:
        rec["status"] = "page_dead"
        return rec
    rec["via"] = via

    img_url = find_image_url(page.text, page.url)
    if not img_url:
        rec["status"] = "no_image_tag"
        return rec
    rec["image_url"] = img_url

    r = get(img_url, fc["timeout"])
    if r is None and via == "live" and fc["use_wayback"]:
        snap = wayback(img_url, fc["timeout"])
        r = get(snap, fc["timeout"]) if snap else None
    if r is None:
        rec["status"] = "image_dead"
        return rec
    try:
        im = Image.open(io.BytesIO(r.content)).convert("RGB")
    except Exception:
        rec["status"] = "image_corrupt"
        return rec
    if min(im.size) < fc["min_image_side"]:
        rec["status"] = "image_too_small"
        return rec
    out = p(f"{cfg['paths']['images']}/{row['id']}.jpg")
    im.save(out, "JPEG", quality=90)
    rec.update(status="ok", image_path=str(out.relative_to(p("."))), width=im.size[0], height=im.size[1])
    return rec


def summarise(cfg: dict) -> dict:
    d = cfg["paths"]["processed"]
    man = p(f"{d}/fnn_manifest.jsonl")
    rows = [json.loads(line) for line in open(man)] if man.exists() else []
    if not rows:
        print("[fnn] manifest empty")
        return {}
    df = pd.DataFrame(rows)
    ok = df[df["status"] == "ok"].copy()
    ok["label"] = ok["fnn_label"].map({"fake": "Fabricated", "real": "Credible"})
    ok[["id", "title", "fnn_label", "label", "site", "image_path", "news_url"]] \
        .to_csv(p(f"{d}/fnn_pairs.csv"), index=False)
    s = {
        "attempted": len(df),
        "usable_pairs": len(ok),
        "usable_rate": round(len(ok) / len(df), 3),
        "status_counts": df["status"].value_counts().to_dict(),
        "usable_by_label": ok["fnn_label"].value_counts().to_dict(),
        "usable_by_site": ok["site"].value_counts().to_dict(),
        "usable_via": ok["via"].value_counts().to_dict(),
    }
    write_json(s, f"{d}/fnn_image_summary.json")
    print(json.dumps(s, indent=2))
    return s


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--limit", type=int, default=None, help="override sample_size")
    ap.add_argument("--summary-only", action="store_true")
    args = ap.parse_args()
    cfg = load_config()
    if args.summary_only:
        summarise(cfg)
        return

    df = load_fnn(p(cfg["fakenewsnet"]["dir"]))
    print(f"[fnn] {len(df)} articles with URL+title | "
          f"{df.groupby(['site','fnn_label']).size().to_dict()}")
    sample = sample_balanced(df, args.limit or cfg["fakenewsnet"]["sample_size"], cfg["seed"])

    man = p(f"{cfg['paths']['processed']}/fnn_manifest.jsonl")
    done = {json.loads(line)["id"] for line in open(man)} if man.exists() else set()
    todo = [r for r in sample.to_dict("records") if r["id"] not in done]
    print(f"[fnn] sample={len(sample)} already_done={len(done)} todo={len(todo)}")

    n = 0
    with ThreadPoolExecutor(cfg["fakenewsnet"]["workers"]) as ex, open(man, "a") as fout:
        futs = [ex.submit(process, r, cfg) for r in todo]
        for fut in as_completed(futs):
            rec = fut.result()
            with _lock:
                fout.write(json.dumps(rec) + "\n")
                fout.flush()
            n += 1
            if n % 50 == 0:
                print(f"[fnn] {n}/{len(todo)}", flush=True)
    summarise(cfg)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/06_train_baseline.py
"""Week-1 baseline: RoBERTa-base, five classes, class-weighted cross-entropy.

This is the "fine-tuned RoBERTa without the proposed modules" baseline from the
evaluation plan. It is also the victim model for TextFooler generation.

Input : data/processed/corpus_{train,valid,test}.csv, corpus_stats.json
Output: runs/baseline/best/            (model + tokenizer, chosen by valid macro-F1)
        runs/baseline/metrics.json     (valid + test: acc, macro-F1, per-class, per-source)
        runs/baseline/confusion_test.csv

Usage : python scripts/06_train_baseline.py [--epochs N] [--smoke]
"""
from __future__ import annotations

import argparse
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch import nn
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import class_names, load_config, p, set_seed, write_json  # noqa: E402


class TextDS(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame, tok, max_len: int):
        self.enc = tok(df["text"].tolist(), truncation=True, max_length=max_len)
        self.y = df["label_id"].astype(int).tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item


class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights: torch.Tensor, **kw):
        super().__init__(*a, **kw)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.functional.cross_entropy(out.logits, labels,
                                           weight=self.class_weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss


def metrics_fn(eval_pred):
    logits, labels = eval_pred
    pred = logits.argmax(-1)
    present = sorted(set(labels.tolist()))
    return {"accuracy": accuracy_score(labels, pred),
            "macro_f1": f1_score(labels, pred, labels=present, average="macro", zero_division=0)}


def report(df: pd.DataFrame, pred: np.ndarray, classes: list[str]) -> dict:
    y = df["label_id"].to_numpy()
    present = sorted(set(y.tolist()))
    r = {
        "n": len(df),
        "accuracy": round(accuracy_score(y, pred), 4),
        "macro_f1": round(f1_score(y, pred, labels=present, average="macro", zero_division=0), 4),
        "classes_in_eval": [classes[i] for i in present],
        "per_class": classification_report(y, pred, labels=list(range(len(classes))),
                                           target_names=classes, output_dict=True, zero_division=0),
        "per_source": {},
    }
    for src, idx in df.groupby("source").groups.items():
        ys, ps = y[df.index.get_indexer(idx)], pred[df.index.get_indexer(idx)]
        pr = sorted(set(ys.tolist()))
        r["per_source"][src] = {"n": len(ys), "accuracy": round(accuracy_score(ys, ps), 4),
                                "macro_f1": round(f1_score(ys, ps, labels=pr, average="macro",
                                                           zero_division=0), 4)}
    return r


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=float, default=None)
    ap.add_argument("--smoke", action="store_true", help="tiny model + 200 rows, CPU-friendly check")
    ap.add_argument("--lr", type=float, default=None)
    ap.add_argument("--run-name", default="baseline", help="output folder under runs/")
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--out-dir", default=None, help="absolute output folder (e.g. Colab local disk instead of Drive)")
    args = ap.parse_args()

    cfg = load_config()
    if args.seed is not None:
        cfg["seed"] = args.seed
    bc = cfg["baseline"]
    set_seed(cfg["seed"])
    classes = class_names()
    d = cfg["paths"]["processed"]
    out_dir = Path(args.out_dir) if args.out_dir else p(f"{cfg['paths']['runs']}/{args.run_name}")
    out_dir.mkdir(parents=True, exist_ok=True)

    train, valid, test = (pd.read_csv(p(f"{d}/corpus_{s}.csv")) for s in ["train", "valid", "test"])
    stats = json.load(open(p(f"{d}/corpus_stats.json")))
    weights = torch.tensor([stats["class_weights"][c] for c in classes], dtype=torch.float)

    model_name = bc["model"]
    if args.smoke:
        model_name = "hf-internal-testing/tiny-random-RobertaForSequenceClassification"
        train, valid, test = train.head(200), valid.head(100), test.head(100)

    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(classes),
        id2label=dict(enumerate(classes)), label2id={c: i for i, c in enumerate(classes)},
        ignore_mismatched_sizes=args.smoke)

    ds = {k: TextDS(v, tok, bc["max_len"]) for k, v in
          [("train", train), ("valid", valid), ("test", test)]}

    targs = TrainingArguments(
        output_dir=str(out_dir / "ckpt"),
        num_train_epochs=args.epochs or bc["epochs"],
        per_device_train_batch_size=bc["batch_size"],
        per_device_eval_batch_size=bc["batch_size"] * 2,
        learning_rate=args.lr or bc["lr"],
        # warmup as an integer step count: works on transformers 4.x and 5.x
        # (5.x removed warmup_ratio)
        warmup_steps=int(math.ceil(len(train) / bc["batch_size"])
                         * (args.epochs or bc["epochs"]) * bc["warmup_ratio"]),
        weight_decay=bc["weight_decay"],
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=bc["fp16"] and torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        seed=cfg["seed"],
    )
    trainer = WeightedTrainer(model=model, args=targs, train_dataset=ds["train"],
                              eval_dataset=ds["valid"], data_collator=DataCollatorWithPadding(tok),
                              compute_metrics=metrics_fn, class_weights=weights)
    trainer.train()
    trainer.save_model(str(out_dir / "best"))
    tok.save_pretrained(str(out_dir / "best"))

    results = {"model": model_name, "class_weights": stats["class_weights"], "config": bc,
               "run_name": args.run_name, "seed": cfg["seed"], "lr": args.lr or bc["lr"], "epochs": args.epochs or bc["epochs"]}
    for name, df in [("valid", valid), ("test", test)]:
        pred = trainer.predict(ds[name]).predictions.argmax(-1)
        results[name] = report(df.reset_index(drop=True), pred, classes)
        print(f"[{args.run_name}] {name}: acc={results[name]['accuracy']} macro_f1={results[name]['macro_f1']} "
              f"per_source={results[name]['per_source']}")
        if name == "test":
            cm = confusion_matrix(df["label_id"], pred, labels=list(range(len(classes))))
            pd.DataFrame(cm, index=classes, columns=classes).to_csv(out_dir / "confusion_test.csv")
            pd.DataFrame({"id": df["id"], "text": df["text"], "label": df["label"],
                          "pred": [classes[i] for i in pred]}).to_csv(out_dir / "test_predictions.csv",
                                                                       index=False)
    write_json(results, out_dir / "metrics.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/07_textfooler_generate.py
"""Generate TextFooler adversarial examples against the week-1 baseline.

Uses src/xfakenet/textfooler.py (the Jin et al. algorithm, no TextAttack needed).

--split train : attacks TRAIN statements until the number of SUCCESSFUL examples
                reaches 15% of the final corpus (70/15/15), with per-class quotas.
                These are the augmentation data for week 3 (gold label kept).
--split test  : attacks a fixed stratified sample of the TEST split. The attack
                success rate is the robustness number for week 4. Never trained on.

Resumable: ids already in the output file are skipped and existing successes count.

Usage: python scripts/07_textfooler_generate.py --split train
       python scripts/07_textfooler_generate.py --split test
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import urllib.request
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, set_seed  # noqa: E402
from xfakenet import textfooler as tf  # noqa: E402

TF_VERSION = 2   # v2: names/numbers protected, antonyms + offensive words blocked


def synonym_source(cfg: dict):
    import zipfile
    tc = cfg["textfooler"]
    path = p(tc["embeddings_path"])
    urls = [tc["embeddings_url"],
            "https://media.githubusercontent.com/media/nmrksic/counter-fitting/master/word_vectors/counter-fitted-vectors.txt.zip"]
    for url in urls:
        if path.exists() and zipfile.is_zipfile(path):
            break
        if path.exists():
            os.remove(path)
        print(f"[textfooler] downloading counter-fitted vectors from {url} ...", flush=True)
        try:
            urllib.request.urlretrieve(url, path)
        except Exception as e:
            print(f"[textfooler] download failed ({e})", flush=True)
    if path.exists() and not zipfile.is_zipfile(path):
        os.remove(path)
        print("[textfooler] download was not a valid zip; using WordNet synonyms instead", flush=True)
    if path.exists():
        print("[textfooler] synonyms: counter-fitted vectors", flush=True)
        return tf.CounterFitted(str(path), k=tc["k"], min_cos=0.5), "counter-fitted"
    import nltk
    nltk.download("wordnet", quiet=True)
    return tf.WordNetSyn(k=tc["k"]), "wordnet"


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", default="train", choices=["train", "test"])
    ap.add_argument("--n", type=int, default=None)
    ap.add_argument("--victim", default=None)
    ap.add_argument("--victim-kind", default="hf", choices=["hf", "xfakenet"],
                    help="hf = Hugging Face classifier (baseline); xfakenet = XFakeNet model (week 3)")
    ap.add_argument("--out", default=None, help="override output file")
    args = ap.parse_args()

    cfg = load_config()
    tc = cfg["textfooler"]
    set_seed(cfg["seed"])

    import nltk
    for r in ["averaged_perceptron_tagger_eng", "averaged_perceptron_tagger", "wordnet"]:
        nltk.download(r, quiet=True)

    syn, syn_name = synonym_source(cfg)
    victim = str(p(args.victim or tc["victim_dir"]))
    if args.victim_kind == "xfakenet":
        from xfakenet.model import Predictor
        predict_fn = Predictor(victim).predict_proba
    else:
        predict_fn = tf.make_predict(victim, cfg["baseline"]["max_len"])
    attacker = tf.TextFooler(
        predict=predict_fn,
        neighbours=syn, similarity=tf.make_similarity(tc["sim_model"]), pos_ok=tf.make_pos_ok(),
        protect=tf.make_protect(), banned=tf.make_banned(),
        sim_threshold=tc["sim_threshold"], max_change=tc["max_change"], query_budget=tc["query_budget"])

    df = pd.read_csv(p(f"{cfg['paths']['processed']}/corpus_{args.split}.csv"))
    df["id"] = df["id"].astype(str)
    out = p(args.out) if args.out else p(tc["out_train"] if args.split == "train" else tc["out_test"])
    prev = [json.loads(line) for line in open(out)] if out.exists() else []
    if prev and prev[0].get("tf_version") != TF_VERSION:
        old = out.with_name(out.stem + ".v1_unfiltered.jsonl")
        out.rename(old)
        print(f"[textfooler] archived old unfiltered output to {old.name}; regenerating", flush=True)
        prev = []
    done = {r["id"] for r in prev}

    if args.split == "train":
        a = cfg["augmentation"]
        target = args.n or round(len(df) * a["textfooler_share"] / a["original_share"])
        frac = df["label"].value_counts(normalize=True)
        quota = {c: round(target * f) for c, f in frac.items()}
        have = pd.Series([r["label"] for r in prev if r["status"] == "success"]).value_counts().to_dict()
        order = df.sample(frac=1.0, random_state=cfg["seed"])        # whole train, shuffled
    else:
        target = args.n or tc["test_n"]
        frac = df["label"].value_counts(normalize=True)
        order = pd.concat([g.sample(min(len(g), round(target * frac[c])), random_state=cfg["seed"])
                           for c, g in df.groupby("label")]).sample(frac=1.0, random_state=cfg["seed"])
        quota, have = None, {}
    todo = order[~order["id"].isin(done)]
    print(f"[textfooler] victim={victim} split={args.split} synonyms={syn_name} target={target} "
          f"already_done={len(done)} successes_so_far={sum(have.values())}", flush=True)

    t0, n, counts = time.time(), 0, {}
    with open(out, "a") as f:
        for row in todo.itertuples(index=False):
            if quota is not None:
                if all(have.get(c, 0) >= q for c, q in quota.items()):
                    break
                if have.get(row.label, 0) >= quota[row.label]:
                    continue
            try:
                res = attacker.attack(row.text, int(row.label_id))
            except Exception as e:  # keep the long job alive
                res = {"status": "error", "error": repr(e)[:200]}
            rec = {"tf_version": TF_VERSION, "id": row.id, "label": row.label, "label_id": int(row.label_id),
                   "source": row.source, "original": row.text, "synonyms": syn_name, **res}
            f.write(json.dumps(rec) + "\n")
            f.flush()
            counts[res["status"]] = counts.get(res["status"], 0) + 1
            if res["status"] == "success":
                have[row.label] = have.get(row.label, 0) + 1
            n += 1
            if n % 50 == 0:
                rate = n / (time.time() - t0)
                print(f"[textfooler] {n} attacked | {counts} | successes {sum(have.values())}/{target} "
                      f"| {rate:.1f}/s", flush=True)
    print(f"[textfooler] finished {counts} | successes by class {have}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/08_clip_cscore.py
"""Module 2: CLIP contradiction score (C-score) and threshold calibration.

C-score = 1 - cos(E_text, E_image), CLIP ViT-B/32, headline vs article image.

There are no human labels for "this image contradicts this headline", so the
threshold is calibrated the standard way: real headline+image pairs are the
ALIGNED class, and the same headlines paired with another article's image
(shuffled within the same site, so politics images stay with politics) are the
MISMATCHED class. The threshold is fitted on a validation split and reported
on a held-out test split.

Input : data/processed/fnn_pairs.csv, data/images/
Output: results/cscore/cscore_pairs.csv     (every aligned + mismatched pair with its C-score)
        results/cscore/cscore_results.json  (thresholds, AUC, accuracy, fake-vs-real check)
        results/cscore/cscore_hist.png
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, set_seed, write_json  # noqa: E402


def derange_within(groups: pd.Series, rng: np.random.Generator) -> np.ndarray:
    """For each row return the index of a DIFFERENT row from the same group."""
    idx = np.arange(len(groups))
    out = np.empty_like(idx)
    for _, g in pd.Series(idx).groupby(groups.values):
        g = g.to_numpy()
        if len(g) < 2:
            out[g] = g  # cannot mismatch a singleton; dropped later
            continue
        out[g] = _derange(g, rng)
    return out


def _derange(g: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    perm = rng.permutation(len(g))
    shifted = np.roll(perm, 1)              # cyclic shift of a random order: no fixed points
    res = np.empty_like(g)
    res[perm] = g[shifted]
    return res


def youden_threshold(scores: np.ndarray, labels: np.ndarray) -> float:
    """Threshold maximising TPR - FPR (labels: 1 = mismatched)."""
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(labels, scores)
    j = tpr - fpr
    t = thr[int(np.argmax(j))]
    return float(t if np.isfinite(t) else np.median(scores))


def eval_at(scores, labels, t) -> dict:
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    pred = (scores > t).astype(int)
    return {"threshold": round(float(t), 4),
            "accuracy": round(accuracy_score(labels, pred), 4),
            "f1_mismatch": round(f1_score(labels, pred, zero_division=0), 4),
            "precision": round(precision_score(labels, pred, zero_division=0), 4),
            "recall": round(recall_score(labels, pred, zero_division=0), 4),
            "flag_rate_aligned": round(float(pred[labels == 0].mean()), 4),
            "flag_rate_mismatched": round(float(pred[labels == 1].mean()), 4)}


def embed(df: pd.DataFrame, cfg: dict):
    import torch
    from PIL import Image
    from transformers import CLIPModel, CLIPProcessor

    name = cfg["cscore"]["clip_model"]
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPModel.from_pretrained(name).to(dev).eval()
    proc = CLIPProcessor.from_pretrained(name)
    T, I = [], []
    bs = cfg["cscore"]["batch_size"]
    for s in range(0, len(df), bs):
        b = df.iloc[s:s + bs]
        imgs = [Image.open(p(x)).convert("RGB") for x in b["image_path"]]
        inp = proc(text=b["title"].tolist(), images=imgs, return_tensors="pt",
                   padding=True, truncation=True, max_length=77).to(dev)
        with torch.no_grad():
            out = model(**inp)   # forward pass gives projected text_embeds/image_embeds in every version
        T.append(torch.nn.functional.normalize(out.text_embeds, dim=-1).cpu().numpy())
        I.append(torch.nn.functional.normalize(out.image_embeds, dim=-1).cpu().numpy())
        print(f"[cscore] embedded {min(s + bs, len(df))}/{len(df)}", flush=True)
    return np.concatenate(T), np.concatenate(I)


def main() -> None:
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split

    cfg = load_config()
    cc = cfg["cscore"]
    set_seed(cfg["seed"])
    rng = np.random.default_rng(cfg["seed"])
    out_dir = cc["out_dir"]

    df = pd.read_csv(p(f"{cfg['paths']['processed']}/fnn_pairs.csv"))
    df = df[df["image_path"].map(lambda x: p(x).exists())].reset_index(drop=True)
    print(f"[cscore] {len(df)} usable pairs | {df.groupby(['site', 'fnn_label']).size().to_dict()}")

    E_t, E_i = embed(df, cfg)
    df["cscore"] = 1.0 - (E_t * E_i).sum(1)

    # mismatched: headline i with image j from the same site (j != i)
    j = derange_within(df["site"], rng)
    keep = j != np.arange(len(df))
    mis = df[keep].copy()
    mis["cscore"] = 1.0 - (E_t[keep] * E_i[j[keep]]).sum(1)
    mis["image_path"] = df["image_path"].to_numpy()[j[keep]]

    df["pair_type"], mis["pair_type"] = "aligned", "mismatched"
    df["is_mismatch"], mis["is_mismatch"] = 0, 1

    # split by HEADLINE so a headline's aligned and mismatched versions stay in the same split
    strat = df["site"] + "_" + df["fnn_label"]
    val_ids, test_ids = train_test_split(df["id"], test_size=0.5, stratify=strat,
                                         random_state=cfg["seed"])
    allp = pd.concat([df, mis], ignore_index=True)
    allp["split"] = np.where(allp["id"].isin(set(val_ids)), "valid", "test")
    allp.to_csv(p(f"{out_dir}/cscore_pairs.csv"), index=False)

    v, t = allp[allp.split == "valid"], allp[allp.split == "test"]
    t_youden = youden_threshold(v["cscore"].to_numpy(), v["is_mismatch"].to_numpy())
    t_pct = float(np.percentile(v.loc[v.is_mismatch == 0, "cscore"], cc["percentile"]))

    def stats(x):
        return {"mean": round(float(x.mean()), 4), "std": round(float(x.std()), 4),
                "min": round(float(x.min()), 4), "max": round(float(x.max()), 4)}

    al = df  # aligned only, for the fake-vs-real check
    res = {
        "clip_model": cc["clip_model"],
        "n_pairs": {"aligned": len(df), "mismatched": len(mis),
                    "valid_headlines": len(val_ids), "test_headlines": len(test_ids)},
        "cscore_distribution": {"aligned": stats(df["cscore"]), "mismatched": stats(mis["cscore"])},
        "test_auc_aligned_vs_mismatched": round(roc_auc_score(t["is_mismatch"], t["cscore"]), 4),
        "thresholds": {
            "youden_on_valid": eval_at(t["cscore"].to_numpy(), t["is_mismatch"].to_numpy(), t_youden),
            f"p{cc['percentile']}_aligned_valid": eval_at(t["cscore"].to_numpy(),
                                                         t["is_mismatch"].to_numpy(), t_pct),
            "proposal_0.62": eval_at(t["cscore"].to_numpy(), t["is_mismatch"].to_numpy(), 0.62),
        },
        "chosen_threshold": round(t_youden, 4),
        "fake_vs_real_aligned": {
            "mean_cscore_fake": round(float(al.loc[al.fnn_label == "fake", "cscore"].mean()), 4),
            "mean_cscore_real": round(float(al.loc[al.fnn_label == "real", "cscore"].mean()), 4),
            "auc_cscore_predicts_fake": round(roc_auc_score((al.fnn_label == "fake").astype(int),
                                                            al["cscore"]), 4),
        },
        "by_site_test_auc": {s: round(roc_auc_score(g["is_mismatch"], g["cscore"]), 4)
                             for s, g in t.groupby("site") if g["is_mismatch"].nunique() == 2},
    }
    write_json(res, f"{out_dir}/cscore_results.json")
    print(pd.Series(res["thresholds"]).to_string())
    print(f"[cscore] test AUC aligned vs mismatched: {res['test_auc_aligned_vs_mismatched']}")
    print(f"[cscore] chosen threshold (Youden, valid): {res['chosen_threshold']}")

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7, 4))
        bins = np.linspace(allp["cscore"].min(), allp["cscore"].max(), 40)
        ax.hist(df["cscore"], bins=bins, alpha=0.6, label="Aligned (real pair)")
        ax.hist(mis["cscore"], bins=bins, alpha=0.6, label="Mismatched (shuffled image)")
        ax.axvline(t_youden, color="k", ls="--", label=f"Threshold {t_youden:.3f}")
        ax.set_xlabel("C-score = 1 - cos(text, image)")
        ax.set_ylabel("Pairs")
        ax.set_title("CLIP ViT-B/32 contradiction score, FakeNewsNet")
        ax.legend()
        fig.tight_layout()
        fig.savefig(p(f"{out_dir}/cscore_hist.png"), dpi=150)
    except Exception as e:  # plotting is a nice-to-have
        print("[cscore] plot skipped:", e)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/09_gemini_paraphrase.py
"""Generate LLM paraphrases of LIAR statements with the Gemini API (augmentation data).

- train: a class-stratified sample of the TRAIN split, sized so paraphrases are
  15% of the final corpus (70% original / 15% TextFooler / 15% paraphrase).
- test : a separate sample of the TEST split, used only in week 4 to measure
  robustness to paraphrased text. Never used for training.

Each paraphrase keeps the original gold label. A paraphrase is rejected if it
changes any number, is identical to the original, or is far longer/shorter.

API key: Colab secret GEMINI_API_KEY, or environment variable GEMINI_API_KEY.
Free key: https://aistudio.google.com/apikey

Resumable: ids already in the output file are skipped.

Usage: python scripts/09_gemini_paraphrase.py --split train
       python scripts/09_gemini_paraphrase.py --split test
"""
from __future__ import annotations

import argparse
import json
import os
import re
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import clean_text, load_config, p  # noqa: E402

PROMPT = """You rewrite short political statements for a research dataset on robust fake-news detection.

Paraphrase each statement below so it reads like a different person said it:
- Keep exactly the same meaning, claim, people, places and every number.
- Do not correct, soften, exaggerate or comment on the claim, even if it is false.
- Change wording and sentence structure; keep roughly the same length.

Return a JSON array with one object per statement, in the same order:
[{{"i": <index>, "paraphrase": "<text>"}}, ...]

Statements:
{items}"""

NUM = re.compile(r"\d+(?:[.,]\d+)*")


def numbers(s: str) -> list[str]:
    return sorted(x.replace(",", "") for x in NUM.findall(s))


def check(orig: str, para: str) -> str | None:
    """Return a rejection reason, or None if the paraphrase is acceptable."""
    if not para or not para.strip():
        return "empty"
    norm = lambda s: re.sub(r"\W+", " ", s.lower()).strip()  # noqa: E731
    if norm(orig) == norm(para):
        return "identical"
    if numbers(orig) != numbers(para):
        return "numbers_changed"
    r = len(para.split()) / max(1, len(orig.split()))
    if r < 0.5 or r > 2.0:
        return "length"
    return None


def get_key() -> str:
    # The notebook puts the key in the environment; Colab's secret API only works in the kernel itself.
    k = os.environ.get("GEMINI_API_KEY", "")
    if not k:
        sys.exit("[paraphrase] No GEMINI_API_KEY in the environment. Run the key cell in the notebook first.")
    return k


def stratified(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    frac = min(1.0, n / len(df))
    parts = [g.sample(max(1, round(len(g) * frac)), random_state=seed) for _, g in df.groupby("label")]
    return pd.concat(parts).sample(frac=1.0, random_state=seed).head(n)


class Gemini:
    def __init__(self, models: list[str], temperature: float):
        from google import genai
        from google.genai import types
        self.client, self.types = genai.Client(api_key=get_key()), types
        self.models, self.temperature, self.model = models, temperature, None

    def call(self, prompt: str) -> str:
        cfg = self.types.GenerateContentConfig(response_mime_type="application/json",
                                               temperature=self.temperature)
        candidates = [self.model] if self.model else self.models
        last = None
        for m in candidates:
            for attempt in range(6):
                try:
                    r = self.client.models.generate_content(model=m, contents=prompt, config=cfg)
                    if self.model is None:
                        self.model = m
                        print(f"[paraphrase] using model {m}", flush=True)
                    return r.text
                except Exception as e:
                    last, msg = e, str(e)
                    if "404" in msg or "not found" in msg.lower() or "NOT_FOUND" in msg:
                        break  # try the next model name
                    wait = min(60, 5 * 2 ** attempt)
                    print(f"[paraphrase] {m}: {msg[:120]} ... retry in {wait}s", flush=True)
                    time.sleep(wait)
        raise RuntimeError(f"All Gemini calls failed: {last}")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", default="train", choices=["train", "test"])
    ap.add_argument("--n", type=int, default=None)
    args = ap.parse_args()

    cfg = load_config()
    pc = cfg["paraphrase"]
    df = pd.read_csv(p(f"{cfg['paths']['processed']}/corpus_{args.split}.csv"))
    if args.n:
        n = args.n
    elif args.split == "train":
        o = cfg["augmentation"]
        n = round(len(df) * o["paraphrase_share"] / o["original_share"])
    else:
        n = pc["test_n"]
    sub = stratified(df, n, cfg["seed"])

    out = p(pc["out_train"] if args.split == "train" else pc["out_test"])
    done = set()
    if out.exists():
        done = {json.loads(line)["id"] for line in open(out)}
    todo = sub[~sub["id"].astype(str).isin(done)]
    print(f"[paraphrase] split={args.split} target={len(sub)} done={len(done)} todo={len(todo)}", flush=True)

    gem = Gemini(pc["models"], pc["temperature"])
    stats = {"ok": 0}
    bs = pc["batch_size"]
    with open(out, "a") as f:
        for s in range(0, len(todo), bs):
            b = todo.iloc[s:s + bs]
            items = "\n".join(f"{i}. {t}" for i, t in enumerate(b["text"]))
            try:
                raw = gem.call(PROMPT.format(items=items))
                got = {int(o["i"]): clean_text(o.get("paraphrase", "")) for o in json.loads(raw)}
            except Exception as e:
                print(f"[paraphrase] batch {s // bs} failed: {e!r}"[:200], flush=True)
                continue  # not written -> retried on the next run
            for i, row in enumerate(b.itertuples(index=False)):
                para = got.get(i, "")
                why = check(row.text, para)
                rec = {"id": str(row.id), "label": row.label, "label_id": int(row.label_id),
                       "source": row.source, "original": row.text, "text": para,
                       "status": "ok" if why is None else f"rejected:{why}", "model": gem.model}
                stats[rec["status"]] = stats.get(rec["status"], 0) + 1
                f.write(json.dumps(rec) + "\n")
            f.flush()
            print(f"[paraphrase] {min(s + bs, len(todo))}/{len(todo)} {stats}", flush=True)
            time.sleep(pc["sleep_s"])
    print(f"[paraphrase] finished {stats}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile docs/label_mapping.md
# Label mapping (Week 1)

**Status:** fixed 20 Oct 2026. Source of truth: `configs/label_mapping.yaml`.

## Output classes

| id | Class | Meaning |
|---|---|---|
| 0 | Fabricated | Claim is false; no factual basis |
| 1 | Misleading | Contains truth but distorts, omits context or exaggerates |
| 2 | Credible | Accurate or substantially accurate |

## LIAR → XFakeNet

| LIAR rating | PolitiFact definition (short) | XFakeNet |
|---|---|---|
| pants-fire | false and ridiculous | Fabricated |
| false | not accurate | Fabricated |
| barely-true | element of truth, ignores critical facts | Misleading |
| half-true | partially accurate, leaves out details | Misleading |
| mostly-true | accurate but needs clarification | Credible |
| true | accurate, nothing significant missing | Credible |

**Rationale.** The split follows PolitiFact's own definitions: the two ratings
with no factual basis become Fabricated, the two that contain truth but
distort it become Misleading, and the two judged accurate become Credible.

**Known limitation.** Neighbouring LIAR ratings (e.g. false vs barely-true) are
hard to separate even for humans, so the class boundaries are noisy.

## Why Satire and Propaganda were dropped

The original plan had five classes. LIAR has no Satire or Propaganda examples,
so these were taken from TSHP-17 (Rashkin et al., 2017). In the first baseline
run, Propaganda scored F1 0.99 on validation but 0.08 on test. The TSHP-17 test
set uses different news websites from training, so the model had learned each
website's writing style rather than propaganda itself; most test propaganda
was labelled Satire. With no reliable source for these two classes in the
time available, the classifier uses three classes.

| Run | Accuracy | Macro-F1 | Note |
|---|---|---|---|
| Five-class (LIAR + TSHP-17) | 0.482 | 0.434 | Propaganda F1 0.08 on test |
| Three-class (LIAR only) | 0.482 | 0.470 | Adopted |

## FakeNewsNet

Binary fake/real → Fabricated/Credible. Used for C-score calibration and
evaluation only, not for classifier training.

## 3. Install packages

In [ ]:
!pip install -q google-genai sentence-transformers nltk pyyaml
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA", torch.cuda.is_available())

## A. Module 2: CLIP contradiction score (C-score)
Aligned = real headline + its own image. Mismatched = same headline + another article's image from the same site. The threshold is fitted on half the headlines and tested on the other half.

In [ ]:
!python scripts/08_clip_cscore.py 2>&1 | grep -v "embedded"
from IPython.display import Image, display
display(Image("results/cscore/cscore_hist.png", width=650))

## B. Gemini paraphrases

In [ ]:
import os
key = os.environ.get("GEMINI_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception as e:
        print("No Colab secret found:", type(e).__name__)
if not key:
    import getpass
    key = getpass.getpass("Paste your Gemini API key (or just press Enter to skip paraphrases): ").strip()
if key:
    os.environ["GEMINI_API_KEY"] = key
    !python scripts/09_gemini_paraphrase.py --split train
    !python scripts/09_gemini_paraphrase.py --split test
else:
    print("Skipped. Add the key and re-run this cell later; nothing else depends on it this week.")

## C. Model v1: learning-rate sweep, best kept as `runs/model_v1`

In [ ]:
import json, os, shutil, yaml
cfg = yaml.safe_load(open("configs/week1.yaml"))
lrs, ep = cfg["model_v1"]["lrs"], cfg["model_v1"]["epochs"]
if not os.path.exists("runs/model_v1/metrics.json"):
    for lr in lrs:
        name = f"v1_lr{lr:g}"
        if os.path.exists(f"runs/{name}/metrics.json"):
            print(name, "already done"); continue
        !python scripts/06_train_baseline.py --lr {lr} --epochs {ep} --run-name {name} 2>&1 | grep --line-buffered -E "^\[v1|Error|error" | tail -5
        shutil.rmtree(f"runs/{name}/ckpt", ignore_errors=True)        # free Drive space
    runs = {f"v1_lr{lr:g}": json.load(open(f"runs/v1_lr{lr:g}/metrics.json")) for lr in lrs}
    best = max(runs, key=lambda k: runs[k]["valid"]["macro_f1"])
    print("Validation macro-F1:", {k: v["valid"]["macro_f1"] for k, v in runs.items()}, "-> best:", best)
    shutil.copytree(f"runs/{best}", "runs/model_v1", dirs_exist_ok=True)
    for k in runs:                                                     # keep metrics, drop extra weights
        shutil.rmtree(f"runs/{k}/best", ignore_errors=True)
    json.dump({k: {"lr": v["lr"], "valid_macro_f1": v["valid"]["macro_f1"], "test_macro_f1": v["test"]["macro_f1"]}
               for k, v in runs.items()}, open("runs/model_v1/sweep.json", "w"), indent=2)
m = json.load(open("runs/model_v1/metrics.json"))
print(f"model_v1 (lr={m['lr']}): test accuracy {m['test']['accuracy']}, macro-F1 {m['test']['macro_f1']}")

## D. TextFooler adversarial examples
Attacks the Week 1 baseline. Training examples stop when 15% of the corpus is reached; the 500 test statements give the robustness number for Week 4. If it is too slow you can stop the cell (■) and run section E; re-running continues where it stopped.

In [ ]:
!python scripts/07_textfooler_generate.py --split train 2>&1 | grep --line-buffered -E "^\[textfooler\]|Error|Traceback"
!python scripts/07_textfooler_generate.py --split test 2>&1 | grep --line-buffered -E "^\[textfooler\]|Error|Traceback"

## E. Week 2 results for your mentor
Re-run this cell any time to refresh the numbers.

In [ ]:
import json, os, shutil
import pandas as pd

def jl(path):
    return pd.DataFrame([json.loads(l) for l in open(path)]) if os.path.exists(path) else pd.DataFrame()

L = ["# XFakeNet – Week 2 results", ""]

# ---- A. C-score
f = "results/cscore/cscore_results.json"
if os.path.exists(f):
    c = json.load(open(f))
    d, th = c["cscore_distribution"], c["thresholds"]
    L += ["## Module 2: CLIP contradiction score (ViT-B/32, FakeNewsNet)", "",
          f"- Pairs: {c['n_pairs']['aligned']} aligned (real headline + own image), "
          f"{c['n_pairs']['mismatched']} mismatched (same headline + another article's image, same site)",
          f"- Mean C-score: aligned {d['aligned']['mean']:.3f} ± {d['aligned']['std']:.3f}, "
          f"mismatched {d['mismatched']['mean']:.3f} ± {d['mismatched']['std']:.3f}",
          f"- **Separation (test ROC-AUC): {c['test_auc_aligned_vs_mismatched']:.3f}**", "",
          "| Threshold (fitted on validation) | Value | Test accuracy | F1 (mismatch) | Flags aligned | Flags mismatched |",
          "|---|---|---|---|---|---|"]
    for k, r in th.items():
        L.append(f"| {k} | {r['threshold']:.3f} | {r['accuracy']:.3f} | {r['f1_mismatch']:.3f} | "
                 f"{r['flag_rate_aligned']*100:.0f}% | {r['flag_rate_mismatched']*100:.0f}% |")
    fr = c["fake_vs_real_aligned"]
    L += ["", f"- Chosen threshold: **{c['chosen_threshold']:.3f}** (Youden's J on validation)",
          f"- Fake vs real articles (own images): mean C-score {fr['mean_cscore_fake']:.3f} vs "
          f"{fr['mean_cscore_real']:.3f}, AUC {fr['auc_cscore_predicts_fake']:.3f}", ""]

# ---- B. paraphrases
for split in ["train", "test"]:
    p = jl(f"data/augment/paraphrase_{split}.jsonl")
    if len(p):
        if split == "train":
            L += ["## Gemini paraphrases", "", "| Split | Generated | Accepted | Rejected (reasons) | Model |",
                  "|---|---|---|---|---|"]
        ok = (p.status == "ok").sum()
        rej = p[p.status != "ok"].status.str.replace("rejected:", "").value_counts().to_dict()
        L.append(f"| {split} | {len(p)} | {ok} ({ok/len(p)*100:.0f}%) | {rej} | {p.model.dropna().iloc[0] if p.model.notna().any() else "-"} |")
if os.path.exists("data/augment/paraphrase_train.jsonl"):
    p = jl("data/augment/paraphrase_train.jsonl"); ex = p[p.status == "ok"].head(2)
    L += [""] + [f"- *{r.original}* → *{r.text}*" for r in ex.itertuples()] + [""]

# ---- C. model v1
f = "runs/model_v1/metrics.json"
if os.path.exists(f):
    m, b = json.load(open(f)), json.load(open("runs/baseline/metrics.json"))
    import yaml
    classes = yaml.safe_load(open("configs/label_mapping.yaml"))["classes"]
    L += ["## Model v1 (RoBERTa-base, class-weighted CE, LIAR test set)", "",
          "| Model | LR | Epochs | Accuracy | Macro-F1 | " + " | ".join(f"F1 {k}" for k in classes) + " |",
          "|---" * (5 + len(classes)) + "|"]
    for name, r in [("Baseline (week 1)", b), ("Model v1", m)]:
        lr = r.get("lr", r["config"]["lr"]); epochs = r.get("epochs", r["config"]["epochs"])
        L.append(f"| {name} | {lr:g} | {epochs:g} | {r['test']['accuracy']:.3f} | {r['test']['macro_f1']:.3f} | "
                 + " | ".join(f"{r['test']['per_class'][k]['f1-score']:.3f}" for k in classes) + " |")
    if os.path.exists("runs/model_v1/sweep.json"):
        s = json.load(open("runs/model_v1/sweep.json"))
        L += ["", "LR sweep (validation macro-F1): " + ", ".join(f"{v['lr']:g} → {v['valid_macro_f1']:.3f}" for v in s.values())]
    L.append("")

# ---- D. TextFooler
t = jl("data/augment/textfooler_train.jsonl")
if len(t):
    vc = t.status.value_counts().to_dict()
    succ = t[t.status == "success"]
    L += ["## TextFooler adversarial examples (victim: week-1 baseline)", "",
          f"- Train: {len(succ)} successful adversarial examples from {len(t)} attacked statements "
          f"({vc}); synonyms: {t.synonyms.iloc[0]}"]
    if len(succ):
        L.append(f"- Average words changed: {succ.words_changed.mean():.1f}, meaning similarity {succ.similarity.mean():.2f}")
        L.append(f"- Successful examples by class: {succ.label.value_counts().to_dict()}")
        r = succ.iloc[0]; L.append(f"- Example: *{r.original}* → *{r.text}*")
tt = jl("data/augment/textfooler_test.jsonl")
if len(tt):
    vc = tt.status.value_counts()
    s, fl, n = vc.get("success", 0), vc.get("failed", 0), len(tt)
    correct = s + fl
    L += [f"- **Test robustness (week-1 baseline):** clean accuracy {correct/n:.3f} on {n} statements → "
          f"accuracy under attack {fl/n:.3f}; attack success rate {s/max(1,correct)*100:.0f}%"]
L.append("")

report = "\n".join(L)
os.makedirs("results", exist_ok=True)
open("results/week2_summary.md", "w").write(report)
from IPython.display import Markdown, display
display(Markdown(report))

os.makedirs("/content/w2", exist_ok=True)
for src, dst in [("results/week2_summary.md", "week2_summary.md"),
                 ("results/cscore/cscore_results.json", "cscore_results.json"),
                 ("results/cscore/cscore_hist.png", "cscore_hist.png"),
                 ("runs/model_v1/metrics.json", "model_v1_metrics.json"),
                 ("runs/model_v1/sweep.json", "model_v1_lr_sweep.json"),
                 ("runs/model_v1/confusion_test.csv", "model_v1_confusion_test.csv")]:
    if os.path.exists(src): shutil.copy(src, "/content/w2/" + dst)
for src in ["data/augment/paraphrase_train.jsonl", "data/augment/paraphrase_test.jsonl",
            "data/augment/textfooler_train.jsonl", "data/augment/textfooler_test.jsonl"]:
    if os.path.exists(src): shutil.copy(src, "/content/w2/")
shutil.make_archive("/content/xfakenet_week2_results", "zip", "/content/w2")
from google.colab import files
files.download("/content/xfakenet_week2_results.zip")